# Data Preprocessing Using Pyspark
## Import Pyspark Libraries and Create Spark Session


In [1]:
import os
os.environ['PYSPARK_PYTHON'] = 'python'

In [2]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, FloatType

In [3]:
spark = SparkSession.builder.appName("DataPreProcessing").master("local[*]").getOrCreate()

## Load/ Import the data

In [4]:
df = spark.read.csv("dataset/employee.csv", header=True, inferSchema=True)

# If you want to define a custom schema instead of inferring it, you can do so like this, meaning you need to replace the 'inferSchema=True' with 'schema=schema' in the read.csv method:
'''
schema = StructType([
    StructField("id", IntegerType(), True),
    StructField("name", StringType(), True),
    StructField("age", IntegerType(), True),
    StructField("salary", FloatType(), True)
])
'''

# json = spark.read.json("dataset/employee.json", schema=schema), parquet = spark.read.parquet("dataset/employee.parquet", schema=schema)

'\nschema = StructType([\n    StructField("id", IntegerType(), True),\n    StructField("name", StringType(), True),\n    StructField("age", IntegerType(), True),\n    StructField("salary", FloatType(), True)\n])\n'

In [5]:
print(df)
df.show()

DataFrame[id: int, name: string, department: string, salary: int, age: int]
+---+-----+-----------+------+----+
| id| name| department|salary| age|
+---+-----+-----------+------+----+
|  1|Alice|Engineering| 85000|  29|
|  2|  Bob|  Marketing| 62000|  34|
|  3|Carol|Engineering| 91000|  41|
|  4|David|         HR|  NULL|  27|
|  5|  Eve|  Marketing| 70000|  38|
|  6|Frank|       NULL| 54000|NULL|
|  7|Grace|Engineering| 95000|  35|
|  8|Henry|         HR| 58000|  45|
|  9|  Ivy|  Marketing| 67000|  31|
| 10| Jack|Engineering| 78000|  28|
+---+-----+-----------+------+----+



In [6]:
df.printSchema()

root
 |-- id: integer (nullable = true)
 |-- name: string (nullable = true)
 |-- department: string (nullable = true)
 |-- salary: integer (nullable = true)
 |-- age: integer (nullable = true)



In [7]:
df.dtypes

[('id', 'int'),
 ('name', 'string'),
 ('department', 'string'),
 ('salary', 'int'),
 ('age', 'int')]

## Check for any null values

In [8]:
from pyspark.sql.functions import col,sum as spark_sum,isnan,when

In [9]:
null_count = df.select([spark_sum(when(col(c).isNull(), 1).otherwise(0)).alias(c) for c in df.columns])

In [10]:
null_count.show()

+---+----+----------+------+---+
| id|name|department|salary|age|
+---+----+----------+------+---+
|  0|   0|         1|     1|  1|
+---+----+----------+------+---+



In [11]:
df.filter(col("salary").isNull()).show()

+---+-----+----------+------+---+
| id| name|department|salary|age|
+---+-----+----------+------+---+
|  4|David|        HR|  NULL| 27|
+---+-----+----------+------+---+



In [12]:
df.filter(col("name").isNull()).show()

+---+----+----------+------+---+
| id|name|department|salary|age|
+---+----+----------+------+---+
+---+----+----------+------+---+



In [13]:
df_clean = df.dropna()
df_clean.show()

+---+-----+-----------+------+---+
| id| name| department|salary|age|
+---+-----+-----------+------+---+
|  1|Alice|Engineering| 85000| 29|
|  2|  Bob|  Marketing| 62000| 34|
|  3|Carol|Engineering| 91000| 41|
|  5|  Eve|  Marketing| 70000| 38|
|  7|Grace|Engineering| 95000| 35|
|  8|Henry|         HR| 58000| 45|
|  9|  Ivy|  Marketing| 67000| 31|
| 10| Jack|Engineering| 78000| 28|
+---+-----+-----------+------+---+



In [14]:
df_filled = df.fillna({"salary": 0, "department": "Unknown"})
df_filled.show()

+---+-----+-----------+------+----+
| id| name| department|salary| age|
+---+-----+-----------+------+----+
|  1|Alice|Engineering| 85000|  29|
|  2|  Bob|  Marketing| 62000|  34|
|  3|Carol|Engineering| 91000|  41|
|  4|David|         HR|     0|  27|
|  5|  Eve|  Marketing| 70000|  38|
|  6|Frank|    Unknown| 54000|NULL|
|  7|Grace|Engineering| 95000|  35|
|  8|Henry|         HR| 58000|  45|
|  9|  Ivy|  Marketing| 67000|  31|
| 10| Jack|Engineering| 78000|  28|
+---+-----+-----------+------+----+



In [15]:
df.describe().show()

+-------+------------------+-----+-----------+-----------------+-----------------+
|summary|                id| name| department|           salary|              age|
+-------+------------------+-----+-----------+-----------------+-----------------+
|  count|                10|   10|          9|                9|                9|
|   mean|               5.5| NULL|       NULL|73333.33333333333|34.22222222222222|
| stddev|3.0276503540974917| NULL|       NULL|14696.93845669907|6.180165405913052|
|    min|                 1|Alice|Engineering|            54000|               27|
|    max|                10| Jack|  Marketing|            95000|               45|
+-------+------------------+-----+-----------+-----------------+-----------------+



In [16]:
df.describe("salary").show()

+-------+-----------------+
|summary|           salary|
+-------+-----------------+
|  count|                9|
|   mean|73333.33333333333|
| stddev|14696.93845669907|
|    min|            54000|
|    max|            95000|
+-------+-----------------+



In [17]:
# Selecting Columns
df.select("name", "salary").show()

+-----+------+
| name|salary|
+-----+------+
|Alice| 85000|
|  Bob| 62000|
|Carol| 91000|
|David|  NULL|
|  Eve| 70000|
|Frank| 54000|
|Grace| 95000|
|Henry| 58000|
|  Ivy| 67000|
| Jack| 78000|
+-----+------+



In [18]:
df.select(col("name"),col("department"), col("salary")).show()

+-----+-----------+------+
| name| department|salary|
+-----+-----------+------+
|Alice|Engineering| 85000|
|  Bob|  Marketing| 62000|
|Carol|Engineering| 91000|
|David|         HR|  NULL|
|  Eve|  Marketing| 70000|
|Frank|       NULL| 54000|
|Grace|Engineering| 95000|
|Henry|         HR| 58000|
|  Ivy|  Marketing| 67000|
| Jack|Engineering| 78000|
+-----+-----------+------+



In [19]:
df.select(col("name"),col("salary"),(col("salary") * 2).alias("salary_with_bonus")).show()

+-----+------+-----------------+
| name|salary|salary_with_bonus|
+-----+------+-----------------+
|Alice| 85000|           170000|
|  Bob| 62000|           124000|
|Carol| 91000|           182000|
|David|  NULL|             NULL|
|  Eve| 70000|           140000|
|Frank| 54000|           108000|
|Grace| 95000|           190000|
|Henry| 58000|           116000|
|  Ivy| 67000|           134000|
| Jack| 78000|           156000|
+-----+------+-----------------+



In [20]:
df.show()

+---+-----+-----------+------+----+
| id| name| department|salary| age|
+---+-----+-----------+------+----+
|  1|Alice|Engineering| 85000|  29|
|  2|  Bob|  Marketing| 62000|  34|
|  3|Carol|Engineering| 91000|  41|
|  4|David|         HR|  NULL|  27|
|  5|  Eve|  Marketing| 70000|  38|
|  6|Frank|       NULL| 54000|NULL|
|  7|Grace|Engineering| 95000|  35|
|  8|Henry|         HR| 58000|  45|
|  9|  Ivy|  Marketing| 67000|  31|
| 10| Jack|Engineering| 78000|  28|
+---+-----+-----------+------+----+



In [21]:
df = df.withColumn("salary_band", when(col("salary") >= 80000, "High").when(col("salary") >= 60000, "Medium").otherwise("Low"))
df.show()

+---+-----+-----------+------+----+-----------+
| id| name| department|salary| age|salary_band|
+---+-----+-----------+------+----+-----------+
|  1|Alice|Engineering| 85000|  29|       High|
|  2|  Bob|  Marketing| 62000|  34|     Medium|
|  3|Carol|Engineering| 91000|  41|       High|
|  4|David|         HR|  NULL|  27|        Low|
|  5|  Eve|  Marketing| 70000|  38|     Medium|
|  6|Frank|       NULL| 54000|NULL|        Low|
|  7|Grace|Engineering| 95000|  35|       High|
|  8|Henry|         HR| 58000|  45|        Low|
|  9|  Ivy|  Marketing| 67000|  31|     Medium|
| 10| Jack|Engineering| 78000|  28|     Medium|
+---+-----+-----------+------+----+-----------+



In [22]:
# df = df.select((col("salary") * 2).alias("adjusted_salary"), col("name"), col("department"))
# this will select only the adjusted salary, name, and department columns, and the original salary column will be dropped from the resulting DataFrame. If you want to keep the original salary column as well, you can include it in the select statement like this:
# df = df.select(col("salary"), (col("salary") * 2).alias("adjusted_salary"), col("name"), col("department"))

In [23]:
# Rows Selection
df.filter(col("salary") > 65000).show()

+---+-----+-----------+------+---+-----------+
| id| name| department|salary|age|salary_band|
+---+-----+-----------+------+---+-----------+
|  1|Alice|Engineering| 85000| 29|       High|
|  3|Carol|Engineering| 91000| 41|       High|
|  5|  Eve|  Marketing| 70000| 38|     Medium|
|  7|Grace|Engineering| 95000| 35|       High|
|  9|  Ivy|  Marketing| 67000| 31|     Medium|
| 10| Jack|Engineering| 78000| 28|     Medium|
+---+-----+-----------+------+---+-----------+



In [24]:
df.filter(col("department") == "Engineering").show()

+---+-----+-----------+------+---+-----------+
| id| name| department|salary|age|salary_band|
+---+-----+-----------+------+---+-----------+
|  1|Alice|Engineering| 85000| 29|       High|
|  3|Carol|Engineering| 91000| 41|       High|
|  7|Grace|Engineering| 95000| 35|       High|
| 10| Jack|Engineering| 78000| 28|     Medium|
+---+-----+-----------+------+---+-----------+



In [25]:
df.where(col("department") == "Engineering").show()
# filter and where are essentially the same in PySpark, and you can use either one based on your preference. Both methods will yield the same result when filtering rows based on a condition.

+---+-----+-----------+------+---+-----------+
| id| name| department|salary|age|salary_band|
+---+-----+-----------+------+---+-----------+
|  1|Alice|Engineering| 85000| 29|       High|
|  3|Carol|Engineering| 91000| 41|       High|
|  7|Grace|Engineering| 95000| 35|       High|
| 10| Jack|Engineering| 78000| 28|     Medium|
+---+-----+-----------+------+---+-----------+



In [26]:
df.where((col("age") < 40) & (col("salary") > 60000)).show()

+---+-----+-----------+------+---+-----------+
| id| name| department|salary|age|salary_band|
+---+-----+-----------+------+---+-----------+
|  1|Alice|Engineering| 85000| 29|       High|
|  2|  Bob|  Marketing| 62000| 34|     Medium|
|  5|  Eve|  Marketing| 70000| 38|     Medium|
|  7|Grace|Engineering| 95000| 35|       High|
|  9|  Ivy|  Marketing| 67000| 31|     Medium|
| 10| Jack|Engineering| 78000| 28|     Medium|
+---+-----+-----------+------+---+-----------+



In [27]:
df.limit(3).show()

+---+-----+-----------+------+---+-----------+
| id| name| department|salary|age|salary_band|
+---+-----+-----------+------+---+-----------+
|  1|Alice|Engineering| 85000| 29|       High|
|  2|  Bob|  Marketing| 62000| 34|     Medium|
|  3|Carol|Engineering| 91000| 41|       High|
+---+-----+-----------+------+---+-----------+



In [28]:
df_1 = df

In [29]:
df_1.show()

+---+-----+-----------+------+----+-----------+
| id| name| department|salary| age|salary_band|
+---+-----+-----------+------+----+-----------+
|  1|Alice|Engineering| 85000|  29|       High|
|  2|  Bob|  Marketing| 62000|  34|     Medium|
|  3|Carol|Engineering| 91000|  41|       High|
|  4|David|         HR|  NULL|  27|        Low|
|  5|  Eve|  Marketing| 70000|  38|     Medium|
|  6|Frank|       NULL| 54000|NULL|        Low|
|  7|Grace|Engineering| 95000|  35|       High|
|  8|Henry|         HR| 58000|  45|        Low|
|  9|  Ivy|  Marketing| 67000|  31|     Medium|
| 10| Jack|Engineering| 78000|  28|     Medium|
+---+-----+-----------+------+----+-----------+



In [30]:
df_1.drop("age").show() 

+---+-----+-----------+------+-----------+
| id| name| department|salary|salary_band|
+---+-----+-----------+------+-----------+
|  1|Alice|Engineering| 85000|       High|
|  2|  Bob|  Marketing| 62000|     Medium|
|  3|Carol|Engineering| 91000|       High|
|  4|David|         HR|  NULL|        Low|
|  5|  Eve|  Marketing| 70000|     Medium|
|  6|Frank|       NULL| 54000|        Low|
|  7|Grace|Engineering| 95000|       High|
|  8|Henry|         HR| 58000|        Low|
|  9|  Ivy|  Marketing| 67000|     Medium|
| 10| Jack|Engineering| 78000|     Medium|
+---+-----+-----------+------+-----------+



In [31]:
df_1.filter(col("department") != "HR").show()

+---+-----+-----------+------+---+-----------+
| id| name| department|salary|age|salary_band|
+---+-----+-----------+------+---+-----------+
|  1|Alice|Engineering| 85000| 29|       High|
|  2|  Bob|  Marketing| 62000| 34|     Medium|
|  3|Carol|Engineering| 91000| 41|       High|
|  5|  Eve|  Marketing| 70000| 38|     Medium|
|  7|Grace|Engineering| 95000| 35|       High|
|  9|  Ivy|  Marketing| 67000| 31|     Medium|
| 10| Jack|Engineering| 78000| 28|     Medium|
+---+-----+-----------+------+---+-----------+



In [32]:
df_1.filter((col("department") != "HR") & (col("department") != "Marketing")).show()

+---+-----+-----------+------+---+-----------+
| id| name| department|salary|age|salary_band|
+---+-----+-----------+------+---+-----------+
|  1|Alice|Engineering| 85000| 29|       High|
|  3|Carol|Engineering| 91000| 41|       High|
|  7|Grace|Engineering| 95000| 35|       High|
| 10| Jack|Engineering| 78000| 28|     Medium|
+---+-----+-----------+------+---+-----------+



In [33]:
df_1.withColumnRenamed("salary", "annual_salary").show()

+---+-----+-----------+-------------+----+-----------+
| id| name| department|annual_salary| age|salary_band|
+---+-----+-----------+-------------+----+-----------+
|  1|Alice|Engineering|        85000|  29|       High|
|  2|  Bob|  Marketing|        62000|  34|     Medium|
|  3|Carol|Engineering|        91000|  41|       High|
|  4|David|         HR|         NULL|  27|        Low|
|  5|  Eve|  Marketing|        70000|  38|     Medium|
|  6|Frank|       NULL|        54000|NULL|        Low|
|  7|Grace|Engineering|        95000|  35|       High|
|  8|Henry|         HR|        58000|  45|        Low|
|  9|  Ivy|  Marketing|        67000|  31|     Medium|
| 10| Jack|Engineering|        78000|  28|     Medium|
+---+-----+-----------+-------------+----+-----------+



In [34]:
# Change the column of entire table
new_names = ["emp_id", "full_name", "dept", "pay", "years", "s_band"]
df_2 = df_1.toDF(*new_names).show()

+------+---------+-----------+-----+-----+------+
|emp_id|full_name|       dept|  pay|years|s_band|
+------+---------+-----------+-----+-----+------+
|     1|    Alice|Engineering|85000|   29|  High|
|     2|      Bob|  Marketing|62000|   34|Medium|
|     3|    Carol|Engineering|91000|   41|  High|
|     4|    David|         HR| NULL|   27|   Low|
|     5|      Eve|  Marketing|70000|   38|Medium|
|     6|    Frank|       NULL|54000| NULL|   Low|
|     7|    Grace|Engineering|95000|   35|  High|
|     8|    Henry|         HR|58000|   45|   Low|
|     9|      Ivy|  Marketing|67000|   31|Medium|
|    10|     Jack|Engineering|78000|   28|Medium|
+------+---------+-----------+-----+-----+------+



In [35]:
df_1.orderBy(col("salary").desc()).show()

+---+-----+-----------+------+----+-----------+
| id| name| department|salary| age|salary_band|
+---+-----+-----------+------+----+-----------+
|  7|Grace|Engineering| 95000|  35|       High|
|  3|Carol|Engineering| 91000|  41|       High|
|  1|Alice|Engineering| 85000|  29|       High|
| 10| Jack|Engineering| 78000|  28|     Medium|
|  5|  Eve|  Marketing| 70000|  38|     Medium|
|  9|  Ivy|  Marketing| 67000|  31|     Medium|
|  2|  Bob|  Marketing| 62000|  34|     Medium|
|  8|Henry|         HR| 58000|  45|        Low|
|  6|Frank|       NULL| 54000|NULL|        Low|
|  4|David|         HR|  NULL|  27|        Low|
+---+-----+-----------+------+----+-----------+



In [36]:
df_1.groupBy("department").agg(F.count("salary").alias("avg_salary")).show()

+-----------+----------+
| department|avg_salary|
+-----------+----------+
|Engineering|         4|
|         HR|         1|
|       NULL|         1|
|  Marketing|         3|
+-----------+----------+



In [37]:
df_2 = df_1.dropna()

In [38]:
df_2.show()

+---+-----+-----------+------+---+-----------+
| id| name| department|salary|age|salary_band|
+---+-----+-----------+------+---+-----------+
|  1|Alice|Engineering| 85000| 29|       High|
|  2|  Bob|  Marketing| 62000| 34|     Medium|
|  3|Carol|Engineering| 91000| 41|       High|
|  5|  Eve|  Marketing| 70000| 38|     Medium|
|  7|Grace|Engineering| 95000| 35|       High|
|  8|Henry|         HR| 58000| 45|        Low|
|  9|  Ivy|  Marketing| 67000| 31|     Medium|
| 10| Jack|Engineering| 78000| 28|     Medium|
+---+-----+-----------+------+---+-----------+



In [39]:
df_2.write.csv("output/employee_clean.csv", header=True, mode="overwrite")

In [40]:
df_2.show()

+---+-----+-----------+------+---+-----------+
| id| name| department|salary|age|salary_band|
+---+-----+-----------+------+---+-----------+
|  1|Alice|Engineering| 85000| 29|       High|
|  2|  Bob|  Marketing| 62000| 34|     Medium|
|  3|Carol|Engineering| 91000| 41|       High|
|  5|  Eve|  Marketing| 70000| 38|     Medium|
|  7|Grace|Engineering| 95000| 35|       High|
|  8|Henry|         HR| 58000| 45|        Low|
|  9|  Ivy|  Marketing| 67000| 31|     Medium|
| 10| Jack|Engineering| 78000| 28|     Medium|
+---+-----+-----------+------+---+-----------+

